# Chuẩn bị dataset

Notebook này tạo dữ liệu processed và split cố định từ `data/raw/`.

**Lưu ý:** notebook này không tự chạy khi mở file. Chỉ khi chọn `Run All` thì các cell ghi file mới tạo hoặc ghi đè dữ liệu trong `data/processed/` và `data/splits/v1/`. Dữ liệu trong `data/raw/` chỉ được đọc, không bị sửa.

## 1. Cấu hình

Các tham số xử lý được đặt ở một chỗ để dễ kiểm tra và tái lập. Ảnh có cạnh ngắn dưới 50 pixel sẽ bị loại. Ảnh hợp lệ được chuyển sang RGB và lưu dưới dạng JPEG.

In [1]:
from collections import Counter, defaultdict
from pathlib import Path
import csv
import hashlib
import json
import random
import shutil

from PIL import Image

PROJECT_ROOT = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / 'src' / 'data' / 'dataset.py').exists()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Không tìm thấy project root')

RAW_ROOT = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_ROOT = PROJECT_ROOT / 'data' / 'processed'
IMAGE_ROOT = PROCESSED_ROOT / 'images'
SPLIT_ROOT = PROJECT_ROOT / 'data' / 'splits' / 'v1'
MIN_SHORT_SIDE = 50
VAL_RATIO = 0.20
SEED = 42
JPEG_QUALITY = 95
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.gif', '.webp'}
LABEL_ALIASES = {
    'apple': 'apple', 'avocado': 'avocado', 'banana': 'banana',
    'cherry': 'cherry', 'kiwi': 'kiwi', 'mango': 'mango',
    'orange': 'orange', 'pinenapple': 'pineapple',
    'pineapple': 'pineapple', 'stawberries': 'strawberries',
    'strawberries': 'strawberries', 'watermelon': 'watermelon',
}
CLASS_NAMES = sorted(set(LABEL_ALIASES.values()))
print('Project root:', PROJECT_ROOT)
print('Raw root:', RAW_ROOT)
print('Output root:', PROCESSED_ROOT)

Project root: d:\vscode\Python\DL\cv\classification
Raw root: d:\vscode\Python\DL\cv\classification\data\raw
Output root: d:\vscode\Python\DL\cv\classification\data\processed


## 2. Quét và kiểm tra dữ liệu raw

Cell này chỉ đọc ảnh, kiểm tra ảnh có mở được không và ghi thông tin metadata vào bộ nhớ. Chưa có file output nào được tạo ở bước này.

In [2]:
def scan_raw_split(split):
    rows = []
    for path in sorted((RAW_ROOT / split).rglob('*')):
        if not path.is_file() or path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue
        raw_label = path.parent.name if split != 'predict' else ''
        label = LABEL_ALIASES.get(raw_label.lower()) if raw_label else ''
        if split != 'predict' and label is None:
            raise ValueError(f'Nhãn chưa có mapping: {raw_label!r}')
        with Image.open(path) as image:
            image.verify()
        with Image.open(path) as image:
            rgb = image.convert('RGB')
            rows.append({
                'raw_path': str(path.relative_to(PROJECT_ROOT)).replace('\\', '/'),
                'label': label,
                'source_split': split,
                'width': image.width,
                'height': image.height,
                'short_side': min(image.size),
                'sha256': hashlib.sha256(rgb.tobytes()).hexdigest(),
                'status': 'candidate',
                'exclusion_reason': '',
                'processed_path': '',
            })
    return rows

raw_rows = {
    split: scan_raw_split(split)
    for split in ('train', 'test', 'predict')
}
print({split: len(rows) for split, rows in raw_rows.items()})

{'train': 2301, 'test': 1025, 'predict': 48}


## 3. Tạo bản processed JPEG RGB

Cell này mới bắt đầu ghi file. Ảnh quá nhỏ bị đánh dấu loại. Ảnh còn lại được lưu trong `data/processed/images/` theo split và nhãn canonical.

In [3]:
if IMAGE_ROOT.exists():
    shutil.rmtree(IMAGE_ROOT)
IMAGE_ROOT.mkdir(parents=True, exist_ok=True)

for rows in raw_rows.values():
    for row in rows:
        if row['short_side'] < MIN_SHORT_SIDE:
            row['status'] = 'excluded'
            row['exclusion_reason'] = 'short_side_below_50px'
            continue
        output_label = row['label'] or 'predict'
        output_path = IMAGE_ROOT / row['source_split'] / output_label / f"{row['sha256'][:16]}.jpg"
        output_path.parent.mkdir(parents=True, exist_ok=True)
        source_path = PROJECT_ROOT / row['raw_path']
        with Image.open(source_path) as image:
            image.convert('RGB').save(output_path, format='JPEG', quality=JPEG_QUALITY)
        row['processed_path'] = str(output_path.relative_to(PROJECT_ROOT)).replace('\\', '/')

print('Processed ảnh:', sum(row['status'] == 'candidate' for rows in raw_rows.values() for row in rows))

Processed ảnh: 3373


## 4. Loại duplicate và tạo split

Giữ bản test đầu tiên theo thứ tự ổn định làm benchmark. Các ảnh train trùng test hoặc trùng nội bộ sẽ bị loại. Phần train còn lại được chia stratified thành train/val theo seed 42.

In [4]:
def active(rows):
    return [row for row in rows if row['status'] == 'candidate']

test_by_hash = defaultdict(list)
for row in active(raw_rows['test']):
    test_by_hash[row['sha256']].append(row)
selected_test = []
for rows in test_by_hash.values():
    selected_test.append(rows[0])
    for duplicate in rows[1:]:
        duplicate['status'] = 'excluded'
        duplicate['exclusion_reason'] = 'duplicate_within_test'

test_hashes = {row['sha256'] for row in selected_test}
for row in raw_rows['train']:
    if row['status'] == 'candidate' and row['sha256'] in test_hashes:
        row['status'] = 'excluded'
        row['exclusion_reason'] = 'duplicate_of_test'

seen_train = set()
for row in raw_rows['train']:
    if row['status'] != 'candidate':
        continue
    if row['sha256'] in seen_train:
        row['status'] = 'excluded'
        row['exclusion_reason'] = 'duplicate_within_train'
    seen_train.add(row['sha256'])

clean_train = active(raw_rows['train'])
selected_test = [row for row in selected_test if row['status'] == 'candidate']
rng = random.Random(SEED)
train_split, val_split = [], []
for label in CLASS_NAMES:
    rows = [row for row in clean_train if row['label'] == label]
    rng.shuffle(rows)
    val_count = max(1, round(len(rows) * VAL_RATIO))
    val_split.extend(rows[:val_count])
    train_split.extend(rows[val_count:])

for row in train_split:
    row['assigned_split'] = 'train'
for row in val_split:
    row['assigned_split'] = 'val'
for row in selected_test:
    row['assigned_split'] = 'test'

print('train/val/test:', len(train_split), len(val_split), len(selected_test))

train/val/test: 1743 436 1020


## 5. Ghi manifest, CSV split và metadata

Các cell dưới đây ghi đường dẫn tương đối, nhãn canonical, mã băm và thông tin cấu hình. Ảnh không được copy vào `data/splits/`; CSV trỏ tới bản JPEG trong `data/processed/images/`.

In [5]:
SPLIT_ROOT.mkdir(parents=True, exist_ok=True)
all_rows = [row for rows in raw_rows.values() for row in rows]
manifest_fields = [
    'raw_path', 'processed_path', 'label', 'source_split', 'sha256',
    'width', 'height', 'short_side', 'status', 'exclusion_reason',
    'assigned_split',
]
with (PROCESSED_ROOT / 'manifest.csv').open('w', newline='', encoding='utf-8') as file:
    writer = csv.DictWriter(file, fieldnames=manifest_fields)
    writer.writeheader()
    writer.writerows({field: row.get(field, '') for field in manifest_fields} for row in all_rows)

def write_split(name, rows):
    with (SPLIT_ROOT / f'{name}.csv').open('w', newline='', encoding='utf-8') as file:
        writer = csv.DictWriter(file, fieldnames=['path', 'label', 'sha256'])
        writer.writeheader()
        for row in sorted(rows, key=lambda item: (item['label'], item['processed_path'])):
            writer.writerow({
                'path': row['processed_path'],
                'label': row['label'],
                'sha256': row['sha256'],
            })

write_split('train', train_split)
write_split('val', val_split)
write_split('test', selected_test)

metadata = {
    'version': 'v1', 'seed': SEED, 'val_ratio': VAL_RATIO,
    'classes': CLASS_NAMES, 'source': 'data/raw',
    'policy': 'keep_test_remove_train',
    'min_short_side_px': MIN_SHORT_SIDE,
    'output_format': 'JPEG', 'output_mode': 'RGB',
    'jpeg_quality': JPEG_QUALITY,
    'counts': {
        'raw_train': len(raw_rows['train']),
        'raw_test': len(raw_rows['test']),
        'raw_predict': len(raw_rows['predict']),
        'train': len(train_split), 'val': len(val_split),
        'test': len(selected_test),
    },
    'excluded': dict(Counter(
        row['exclusion_reason'] for row in all_rows
        if row['status'] == 'excluded'
    )),
}
(SPLIT_ROOT / 'metadata.json').write_text(
    json.dumps(metadata, indent=2), encoding='utf-8'
)
print(json.dumps(metadata, indent=2, ensure_ascii=False))

{
  "version": "v1",
  "seed": 42,
  "val_ratio": 0.2,
  "classes": [
    "apple",
    "avocado",
    "banana",
    "cherry",
    "kiwi",
    "mango",
    "orange",
    "pineapple",
    "strawberries",
    "watermelon"
  ],
  "source": "data/raw",
  "policy": "keep_test_remove_train",
  "min_short_side_px": 50,
  "output_format": "JPEG",
  "output_mode": "RGB",
  "jpeg_quality": 95,
  "counts": {
    "raw_train": 2301,
    "raw_test": 1025,
    "raw_predict": 48,
    "train": 1743,
    "val": 436,
    "test": 1020
  },
  "excluded": {
    "duplicate_of_test": 116,
    "duplicate_within_train": 5,
    "short_side_below_50px": 1,
    "duplicate_within_test": 5
  }
}


## 6. Kiểm tra kết quả

Không coi việc notebook chạy xong là đủ. Cell này kiểm tra file tồn tại, ảnh đều là JPEG RGB, cạnh ngắn đạt ngưỡng, đủ lớp và không có mã băm trùng giữa các split.

In [6]:
split_rows = {}
for name in ('train', 'val', 'test'):
    with (SPLIT_ROOT / f'{name}.csv').open(encoding='utf-8-sig') as file:
        split_rows[name] = list(csv.DictReader(file))
    assert split_rows[name], f'Split rỗng: {name}'
    assert {row['label'] for row in split_rows[name]} == set(CLASS_NAMES)
    for row in split_rows[name]:
        path = PROJECT_ROOT / row['path']
        assert path.exists(), path
        with Image.open(path) as image:
            assert image.format == 'JPEG'
            assert image.mode == 'RGB'
            assert min(image.size) >= MIN_SHORT_SIDE

for left, right in (('train', 'val'), ('train', 'test'), ('val', 'test')):
    left_hashes = {row['sha256'] for row in split_rows[left]}
    right_hashes = {row['sha256'] for row in split_rows[right]}
    assert not left_hashes & right_hashes, f'Duplicate: {left}/{right}'

print('Kiểm tra dataset: OK')

Kiểm tra dataset: OK
